In [ ]:
# !gcloud auth application-default login --no-browser

In [ ]:
# Import required libraries
import gc
import logging
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, last, sum as spark_sum, count as sql_count, lag, coalesce, regexp_extract, lower, lit, row_number, collect_list, broadcast, explode, array, struct, monotonically_increasing_id, udf
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, FloatType, IntegerType
from typing import Iterator, Tuple, List, Dict
from pyspark.sql.window import Window
from pyspark import TaskContext
from sentence_transformers import SentenceTransformer
import torch
import torch.nn.functional as Fnn

In [ ]:
# Clear any existing GPU memory
torch.cuda.empty_cache()
gc.collect()

# Set PyTorch memory allocation strategy
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# import shutil
# import os

# spark_temp_path = "/content/spark-temp"
# if os.path.exists(spark_temp_path):
#     shutil.rmtree(spark_temp_path)
#     print(f"✓ Removed {spark_temp_path}")

# !df -h /content/

In [ ]:
# import os

# credentials_path = '/content/.config/application_default_credentials.json'
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = credentials_path
# Set Java home explicitly
# os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'  # Adjust path as needed
spark = SparkSession.builder \
    .appName("generate_ext_user_engagement_dataset") \
    .master("local[16]") \
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.43.1,"
        "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.30"
    ) \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.google.cloud.auth.application_default_credentials.enabled", "true") \
    .config("spark.driver.memory", "100g") \
    .config("spark.driver.memoryOverhead", "16g") \
    .config("spark.executor.memory", "100g") \
    .config("spark.executor.memoryOverhead", "16g") \
    .config("spark.executor.cores", "24") \
    .config("spark.executor.instances", "2") \
    .config("spark.driver.maxResultSize", "20g") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "3") \
    .config("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.local.dir", "/content/spark-temp") \
    .config("hadoop.tmp.dir", "/content/hadoop-tmp") \
    .config("spark.sql.warehouse.dir", "/content/spark-warehouse") \
    .config("spark.worker.cleanup.enabled", "true") \
    .config("spark.worker.cleanup.interval", "900") \
    .config("spark.cleaner.referenceTracking", "true") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.kryoserializer.buffer.max", "1024m") \
    .config("spark.driver.extraJavaOptions", "-Djava.io.tmpdir=/content/tmp") \
    .config("spark.executor.extraJavaOptions", "-Djava.io.tmpdir=/content/tmp") \
    .getOrCreate()

print("Spark Session initialized successfully!")
print(f"Spark Version: {spark.version}")

Spark Session initialized successfully!
Spark Version: 4.0.1


In [ ]:
# Configuration
PROJECT_ID = "honey-production"  # Replace with your GCP project ID
DATASET = "sdata_events_partitioned"       # Replace with your dataset name

# Date parameters (modify as needed)
ANALYSIS_START_DATE = "2025-12-08"
ANALYSIS_END_DATE = "2025-12-08"

print(f"Querying data from {ANALYSIS_START_DATE} to {ANALYSIS_END_DATE}")

Querying data from 2025-12-08 to 2025-12-08


In [ ]:
# Create temp directories on local SSD
import os
os.makedirs("/content/spark-temp", exist_ok=True)
os.makedirs("/content/spark-warehouse", exist_ok=True)
os.makedirs("/content/hadoop-tmp", exist_ok=True)

In [ ]:
page_detected_sql_query = f"""
SELECT distinct
    id,
    ts,
    timestamp,
    store.id AS storeid,
    store.name AS storename,
    store.country AS storecountry,
    store.session_id AS storesessionid,
    extension.screenview_id AS screenviewid,
    page_title,
    page_type,
    user_id,
    session_id,
    referrer_url
FROM `{PROJECT_ID}.{DATASET}.ext_page_detected`
WHERE timestamp IS NOT NULL
AND user_id IS NOT NULL
AND session_id IS NOT NULL
AND extension.screenview_id IS NOT NULL
AND DATE(timestamp) >= '{ANALYSIS_START_DATE}'
AND DATE(timestamp) <= '{ANALYSIS_END_DATE}'
ORDER BY session_id DESC, user_id DESC, ts DESC
"""
# Execute query and create DataFrame
print("Executing BigQuery SQL...")
page_detected_df = spark.read \
    .format("bigquery") \
    .option("query", page_detected_sql_query) \
    .option("project", PROJECT_ID) \
    .option("viewsEnabled", "true") \
    .option("parallelism", "200") \
    .option("materializationProject", PROJECT_ID) \
    .option("materializationDataset", "new_temp_materialization") \
    .option("dataset", "US") \
    .load()
# print(f"\n=== SAMPLE DATA (First 50 rows) ===")
# page_detected_df.show(50, truncate=False)

Executing BigQuery SQL...


In [ ]:
import re
def extract_shein_sc_query_term(search_string):
    """
    Extract the query term from sc=<queryterm> in the search string.
    """
    if not search_string:
        return None

    # Pattern: sc= followed by anything until the next backtick or end of string
    match = re.search(r'sc=([^`]+)', search_string)
    if match:
        return match.group(1)
    return None

In [ ]:
from urllib.parse import urlparse, parse_qs
import pandas as pd

def extract_search_query(referrer_url):
    """
    Extract the search query term from a referrer URL based on known key variants for different merchants.
    :param referrer_url: Referrer URL string.
    :return: Extracted search query term or None if no match is found.
    """
    # Ensure referrer_url is a string; handle NaN values which are floats
    if pd.isna(referrer_url):
        return None

    # List of key variants used by merchants to denote search query terms
    search_query_keys = [
        'keywords',                # Amazon
        '_skw',                    # eBay
        'src_identifier',          # Shein
        'ga_search_query',         # Etsy
        'search_key',              # Temu
        'searchTerm',              # Lowe's
        'searchText',              # Old Navy
        'q',                       # Etsy, Walmart, REI
        'kw',                      # StubHub
        'keyword',                 # Costco, eBay
        'search'                  # BestBuy, Menards, ULTA Beauty
    ]

    # Parse the referrer URL to extract the query string parameters
    parsed_url = urlparse(referrer_url)
    query_params = parse_qs(parsed_url.query)

    # Check for search query keys in the query parameters
    for key in search_query_keys:
        if key in query_params:
            # Return the first matching value for the key
            sanitized_query_term = query_params[key][0]
            if sanitized_query_term:
                sanitized_query_term = sanitized_query_term.strip()
                if sanitized_query_term.startswith('"') and sanitized_query_term.endswith('"'):
                    sanitized_query_term = sanitized_query_term[1:-1]
                sanitized_query_term = sanitized_query_term.replace('+', ' ')
                sanitized_query_term = re.sub(r'\s+', ' ', sanitized_query_term)
                # sanitized_query_term = sanitized_query_term.lower()
                sanitized_query_term = sanitized_query_term.strip()
                #Do a SHEIN specific Search query term extraction
                if key == 'src_identifier':
                    if 'src_module' in query_params and query_params['src_module'][0] == 'search':
                        sanitized_query_term = extract_shein_sc_query_term(sanitized_query_term)
                    else:
                        sanitized_query_term = None
            return sanitized_query_term
            # return query_params[key][0]  # Assuming the first value is the desired search term

    # If no matching key is found, return None
    return None

In [ ]:
# Step 1: Filter for specific page_types
allowed_page_types = ['UNSUPPORTED', 'PRODUCT', 'SEARCH', 'CART_PRODUCT', 'CHECKOUT_CONFIRM']
ATC_CHECKOUT_PAGE_TYPE = ['CART_PRODUCT', 'CHECKOUT_CONFIRM']

page_detected_df_filtered = page_detected_df.filter(F.col('page_type').isin(allowed_page_types))

# Register UDF
extract_search_query_udf = F.udf(extract_search_query, StringType())

# Apply transformations
page_detected_df_final = page_detected_df_filtered.withColumn('search_query_term', extract_search_query_udf(F.col('referrer_url')))

# # Show results
# page_detected_df_final.show(50, truncate=False)
# print(f"Total records after filtering: {page_detected_df_final.count()}")

# Filter rows with non-null search_query_term
filtered_click_df = page_detected_df_final.filter(F.col('search_query_term').isNotNull())

# Get unique user_id, session_id combinations from filtered_click_df
filtered_click_df_merge_fields = filtered_click_df.select('user_id', 'session_id').distinct()

# Filter rows with page_type in ATC_CHECKOUT_PAGE_TYPE
atc_checkout_df = page_detected_df_final.filter(F.col('page_type').isin(ATC_CHECKOUT_PAGE_TYPE))

# Filter rows where search_query_term is null and page_type is 'PRODUCT'
product_non_search_query_terms_df = page_detected_df_final.filter(
    F.col('search_query_term').isNull() &
    (F.col('page_type') == 'PRODUCT')
)

# Union the two dataframes
atc_checkout_df = atc_checkout_df.union(product_non_search_query_terms_df)

# Inner join and select specific columns
filtered_atc_checkout_df = atc_checkout_df.join(
    filtered_click_df_merge_fields,
    on=['user_id', 'session_id'],
    how='inner'
).select(
    'id', 'ts', 'timestamp', 'storeid', 'storename', 'storecountry',
    'storesessionid', 'screenviewid', 'page_title', 'page_type', 'user_id',
    'session_id', 'referrer_url', 'search_query_term'
).distinct()

# Final union
filtered_click_atc_checkout_df = filtered_click_df.union(filtered_atc_checkout_df)
filtered_click_atc_checkout_df = filtered_click_atc_checkout_df.withColumn(
    "is_engagement_verified",
    F.when(F.col("search_query_term").isNotNull(), 1).otherwise(0)
)
# filtered_click_atc_checkout_df.show(100, truncate=False)
# print(f"Total records after filtering for user sessions with at least one search query term: {filtered_click_atc_checkout_df.count()}")

In [ ]:
num_partitions = 800  # Adjust based on: num_cores * 2-4

atc_products_sql_query = f"""
SELECT distinct
extension.screenview_id AS screenview_id,
product.parent_id AS parentid, product.name AS productname, product.url AS producturl, product.total_price AS totalprice, product.sku AS productsku,
user_id AS userid, session_id AS sessionid
FROM `{PROJECT_ID}.{DATASET}.ext_cart_products`, UNNEST(cart.products) AS product
WHERE timestamp IS NOT NULL
AND user_id IS NOT NULL
AND session_id IS NOT NULL
AND extension.screenview_id IS NOT NULL
AND DATE(timestamp) >= '{ANALYSIS_START_DATE}'
AND DATE(timestamp) <= '{ANALYSIS_END_DATE}'
"""
# Execute query and create DataFrame
print("Executing BigQuery SQL...")
atc_products_df = spark.read \
    .format("bigquery") \
    .option("query", atc_products_sql_query) \
    .option("project", PROJECT_ID) \
    .option("viewsEnabled", "true") \
    .option("parallelism", "200") \
    .load()

# print(f"\n=== SAMPLE DATA (First 20 rows) ===")
# atc_products_df.show(50, truncate=False)

atc_products_df = atc_products_df.repartition(
    num_partitions,
    "userid", "sessionid", "screenview_id"
)

filtered_click_atc_checkout_df = filtered_click_atc_checkout_df.repartition(
    num_partitions,
    "user_id", "session_id", "screenviewid"
)

# Force materialization
# atc_products_df.count()
# filtered_click_atc_checkout_df.count()

# Define the join condition
join_condition = [
    filtered_click_atc_checkout_df.user_id == atc_products_df.userid,
    filtered_click_atc_checkout_df.session_id == atc_products_df.sessionid,
    filtered_click_atc_checkout_df.screenviewid == atc_products_df.screenview_id
]

# Perform the join
joined_df = filtered_click_atc_checkout_df.join(atc_products_df, on=join_condition, how='left')

# Update the page_title field and add is_engagement_verified
result_df = joined_df.withColumn(
    "page_title",
    F.when(F.col("productname").isNotNull(), F.col("productname")).otherwise(F.col("page_title"))
).withColumn(
    "is_engagement_verified",
    F.when(F.col("productname").isNotNull(), 1).otherwise(F.col("is_engagement_verified"))
)

# Select fields only from `filtered_click_atc_checkout_df`
engagement_click_atc_df_filtered = result_df.select(*[F.col(field) for field in filtered_click_atc_checkout_df.columns])

# Show the final dataframe
# engagement_click_atc_df_filtered.show(100, truncate=False)

Executing BigQuery SQL...


In [ ]:
checkout_products_sql_query = f"""
SELECT distinct
extension.screenview_id AS screenview_id,
product.parent_id AS parentid, product.name AS productname, product.url AS producturl, product.total_price AS totalprice, product.sku AS productsku,
user_id AS userid, session_id AS sessionid
FROM `{PROJECT_ID}.{DATASET}.ext_checkout_products`, UNNEST(cart.products) AS product
WHERE timestamp IS NOT NULL
AND user_id IS NOT NULL
AND session_id IS NOT NULL
AND extension.screenview_id IS NOT NULL
AND DATE(timestamp) >= '{ANALYSIS_START_DATE}'
AND DATE(timestamp) <= '{ANALYSIS_END_DATE}'
"""
# Execute query and create DataFrame
print("Executing BigQuery SQL...")
checkout_products_df = spark.read \
    .format("bigquery") \
    .option("query", checkout_products_sql_query) \
    .option("project", PROJECT_ID) \
    .option("viewsEnabled", "true") \
    .option("parallelism", "200") \
    .load()


checkout_products_df = checkout_products_df.repartition(
    num_partitions,
    "userid", "sessionid", "screenview_id"
)
engagement_click_atc_df_filtered = engagement_click_atc_df_filtered.repartition(
    num_partitions,
    "user_id", "session_id", "screenviewid"
)
# Force materialization
# checkout_products_df.count()
# engagement_click_atc_df_filtered.count()


# print(f"\n=== SAMPLE DATA (First 50 rows) ===")
# checkout_products_df.show(50, truncate=False)
# Define the join condition
join_condition = [
    engagement_click_atc_df_filtered.user_id == checkout_products_df.userid,
    engagement_click_atc_df_filtered.session_id == checkout_products_df.sessionid,
    engagement_click_atc_df_filtered.screenviewid == checkout_products_df.screenview_id
]

# Perform the join
joined_df = engagement_click_atc_df_filtered.join(checkout_products_df, on=join_condition, how='left')

# Update the page_title field and add is_engagement_verified
result_df = joined_df.withColumn(
    "page_title",
    F.when(F.col("productname").isNotNull(), F.col("productname")).otherwise(F.col("page_title"))
).withColumn(
    "is_engagement_verified",
    F.when(F.col("productname").isNotNull(), 1).otherwise(F.col("is_engagement_verified"))
)

# Select fields only from `filtered_click_atc_checkout_df`
all_engagement_filtered_df = result_df.select(*[F.col(field) for field in engagement_click_atc_df_filtered.columns])
all_engagement_filtered_ordered_df = all_engagement_filtered_df.orderBy(['session_id', 'user_id', 'ts'], ascending=False)
all_engagement_filtered_ordered_df_with_date = all_engagement_filtered_ordered_df.withColumn("date", F.to_date(all_engagement_filtered_ordered_df["timestamp"]))
all_engagement_filtered_ordered_df_with_date = all_engagement_filtered_ordered_df_with_date.repartition(
    num_partitions,
    "userid", "sessionid")

Executing BigQuery SQL...


In [ ]:
all_engagement_tier_0_df = all_engagement_filtered_ordered_df_with_date.filter(col("search_query_term").isNotNull() & (col("page_title").isNotNull()))
all_engagement_tier_0_df = all_engagement_tier_0_df.filter(F.length(F.trim(F.col("search_query_term"))) >= 2)

In [ ]:
refurlpattern = r'/search|search/'
titlepattern = r'Search Result'

# Second filter: exclude rows where referrer_url contains the pattern
all_engagement_tier_0_df = all_engagement_tier_0_df.filter(
    ~col('referrer_url').rlike(refurlpattern) | col('referrer_url').isNull()
)

# Third filter: exclude rows where page_title contains the pattern (case-insensitive)
all_engagement_tier_0_df = all_engagement_tier_0_df.filter(
    ~col('page_title').rlike('(?i)' + titlepattern) | col('page_title').isNull()
)

In [ ]:
all_engagement_tier_0_df_cached = all_engagement_tier_0_df.cache()
print(all_engagement_tier_0_df_cached.count())

3851957


In [ ]:
# Define window spec partitioned by user_id and session_id, ordered by timestamp
window_spec = Window.partitionBy("user_id", "session_id").orderBy("ts")

# Step 1: Identify source rows (where search_query_term can be propagated FROM)
df = all_engagement_filtered_ordered_df_with_date.withColumn(
    "is_source",
    when(
        (col("search_query_term").isNotNull()) &
        (col("page_type").isin(['PRODUCT', 'UNSUPPORTED', 'SEARCH'])),
        1
    ).otherwise(0)
)

# Step 2: Create continuous storesessionid blocks
# Detect when storesessionid changes from previous row
df = df.withColumn(
    "prev_storesessionid",
    lag("storesessionid", 1).over(window_spec)
)

df = df.withColumn(
    "storesession_changed",
    when(
        (col("prev_storesessionid").isNull()) |
        (col("storesessionid") != col("prev_storesessionid")),
        1
    ).otherwise(0)
)

# Create a unique continuous storesessionid block identifier
df = df.withColumn(
    "storesession_block_id",
    spark_sum("storesession_changed").over(
        window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
)

# Step 3: Check if CHECKOUT_CONFIRM exists in each continuous storesessionid block
df = df.withColumn(
    "has_checkout_confirm",
    spark_sum(when(col("page_type") == "CHECKOUT_CONFIRM", 1).otherwise(0)).over(
        Window.partitionBy("user_id", "session_id", "storesession_block_id")
    )
)

# Step 4: Create stop event based on priority within each storesessionid block
df = df.withColumn(
    "is_stop_event",
    when(
        col("has_checkout_confirm") > 0,
        when(col("page_type") == "CHECKOUT_CONFIRM", 1).otherwise(0)
    ).otherwise(
        when(col("page_type") == "CART_PRODUCT", 1).otherwise(0)
    )
)

# Step 5: Lag the stop event to mark the row AFTER as a boundary
window_spec_block = Window.partitionBy("user_id", "session_id", "storesession_block_id").orderBy("ts")

df = df.withColumn(
    "prev_is_stop_event",
    lag("is_stop_event", 1, 0).over(window_spec_block)
)

# Step 6: Create segment_id by cumulative sum of prev_is_stop_event within each storesessionid block
df = df.withColumn(
    "segment_id",
    spark_sum("prev_is_stop_event").over(
        window_spec_block.rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
)

# Step 7: Forward fill search_query_term within each segment of each continuous storesessionid block
window_spec_segment = Window.partitionBy(
    "user_id", "session_id", "storesession_block_id", "segment_id"
).orderBy("ts")

df = df.withColumn(
    "propagated_search_query_term",
    last(when(col("is_source") == 1, col("search_query_term")), ignorenulls=True)
        .over(window_spec_segment.rowsBetween(Window.unboundedPreceding, Window.currentRow))
)

# Step 8: Apply derivation logic only to target page_types
df = df.withColumn(
    "is_search_query_term_derived",
    when(
        (col("page_type").isin(['PRODUCT', 'UNSUPPORTED', 'CART_PRODUCT', 'CHECKOUT_CONFIRM'])) &
        (col("search_query_term").isNull()) &
        (col("propagated_search_query_term").isNotNull()),
        1
    ).otherwise(0)
)

# Step 9: Update search_query_term with propagated value where derived
df = df.withColumn(
    "search_query_term",
    when(
        col("is_search_query_term_derived") == 1,
        col("propagated_search_query_term")
    ).otherwise(col("search_query_term"))
)

# Step 10: Clean up helper columns
df = df.drop(
    "is_source",
    "prev_storesessionid",
    "storesession_changed",
    "storesession_block_id",
    "has_checkout_confirm",
    "is_stop_event",
    "prev_is_stop_event",
    "segment_id",
    "propagated_search_query_term"
)

all_engagement_tier_1_df = df.orderBy(
    ['session_id', 'user_id', 'ts'], ascending=False
)

all_engagement_tier_1_df = all_engagement_tier_1_df.repartition(
    num_partitions,
    "user_id", "session_id"
)

In [ ]:
# Step 1: Filter CART_PRODUCT events
cart_products = all_engagement_tier_1_df.filter(col("page_type") == "CART_PRODUCT")

# Step 2: Identify source events (events with search_query_term to propagate)
source_events = cart_products.filter(
    (col("is_search_query_term_derived") == 1) &
    (col("search_query_term").isNotNull())
).select(
    col("user_id").alias("src_user_id"),
    col("session_id").alias("src_session_id"),
    col("screenviewid").alias("src_screenviewid"),
    col("search_query_term").alias("src_search_query_term")
).distinct()

# Step 3: Identify target events (CART_PRODUCT events that need updating)
target_events = cart_products.filter(
    ~((col("is_search_query_term_derived") == 1) & (col("search_query_term").isNotNull()))
).alias("target")

# Step 4: Perform the join and update
use_broadcast = source_events.count() < 10000000  # Adjust threshold as needed

if use_broadcast:
    print("Using broadcast join for source events")
    source_events = F.broadcast(source_events)

updated_cart_products = target_events.join(
    source_events,
    (col("target.user_id") == col("src_user_id")) &
    (col("target.session_id") == col("src_session_id")) &
    (col("target.screenviewid") == col("src_screenviewid")),
    "left"
).select(
    col("target.id"),
    col("target.ts"),
    col("target.timestamp"),
    col("target.storeid"),
    col("target.storename"),
    col("target.storecountry"),
    col("target.storesessionid"),
    col("target.screenviewid"),
    col("target.page_title"),
    col("target.page_type"),
    col("target.user_id"),
    col("target.session_id"),
    col("target.referrer_url"),
    # Update search_query_term if matched
    when(col("src_search_query_term").isNotNull(), col("src_search_query_term"))
    .otherwise(col("target.search_query_term")).alias("search_query_term"),
    col("target.is_engagement_verified"),
    col("target.date"),
    # Update is_search_query_term_derived if matched
    when(col("src_search_query_term").isNotNull(), 1)
    .otherwise(col("target.is_search_query_term_derived")).alias("is_search_query_term_derived")
)

# Step 5: Get the original source CART_PRODUCT events (THIS WAS MISSING!)
source_cart_products = cart_products.filter(
    (col("is_search_query_term_derived") == 1) &
    (col("search_query_term").isNotNull())
)

# Step 6: Union all CART_PRODUCT events (updated targets + original sources)
all_cart_products = updated_cart_products.unionByName(source_cart_products)

# Step 7: Get non-CART_PRODUCT events
non_cart_products = df.filter(col("page_type") != "CART_PRODUCT")

# Step 8: Combine everything
all_engagement_tier_1_df = all_cart_products.unionByName(non_cart_products)

all_engagement_tier_1_df = all_engagement_tier_1_df.repartition(
    num_partitions,
    "user_id", "session_id")

Using broadcast join for source events


In [ ]:
# Define window specification: partition by user session, order by timestamp
window_spec = Window.partitionBy("user_id", "session_id").orderBy("ts").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Step 1: Identify Amazon stores (case-insensitive check)
df_with_amazon_flag = all_engagement_tier_1_df.withColumn(
    "is_amazon_store",
    when(lower(col("storename")).contains("amazon"), True).otherwise(False)
)

# Step 2: Capture the last Amazon PRODUCT page_title for each event in the session
df_with_last_product = df_with_amazon_flag.withColumn(
    "last_amazon_product_page_title",
    last(
        when(
            (col("is_amazon_store") == True) & (col("page_type") == "PRODUCT"),
            col("page_title")
        ),
        ignorenulls=True
    ).over(window_spec)
)

# Step 3: Update page_title in-place for Amazon CHECKOUT_CONFIRM events
all_engagement_tier_1_df = df_with_last_product.withColumn(
    "page_title",
    when(
        (col("is_amazon_store") == True) &
        (col("page_type") == "CHECKOUT_CONFIRM") &
        col("last_amazon_product_page_title").isNotNull(),
        col("last_amazon_product_page_title")
    ).otherwise(col("page_title"))
).drop("is_amazon_store", "last_amazon_product_page_title")

In [ ]:
refurlpattern = r'/search|search/'
titlepattern = r'Search Result'

# Second filter: exclude rows where referrer_url contains the pattern
all_engagement_tier_1_df = all_engagement_tier_1_df.filter(
    ~col('referrer_url').rlike(refurlpattern) | col('referrer_url').isNull()
)

# Third filter: exclude rows where page_title contains the pattern (case-insensitive)
all_engagement_tier_1_df = all_engagement_tier_1_df.filter(
    ~col('page_title').rlike('(?i)' + titlepattern) | col('page_title').isNull()
)

In [ ]:
all_engagement_tier_1_df = all_engagement_tier_1_df.filter(col("search_query_term").isNotNull() & (col("page_title").isNotNull()))
all_engagement_tier_1_df = all_engagement_tier_1_df.filter(F.length(F.trim(F.col("search_query_term"))) >= 2)
all_engagement_tier_1_df = all_engagement_tier_1_df.orderBy(['session_id', 'user_id', 'ts'], ascending=False)

In [ ]:
# # GCS bucket path
# gcs_path = "gs://chanderiyer/projects/honeysearchbenchmarks/datasets/tiered_user_engagement/sdata_search_query_derived_tier_1_v0"

# # Loop through each partition and coalesce into a single file for each date subfolder
# for date in all_engagement_tier_1_df.select("date").distinct().collect():
#     single_date = date["date"]
#     # Filter by the specific date
#     filtered_df = all_engagement_tier_1_df.filter(F.col("date") == single_date)
#     # Coalesce the DataFrame to ensure a single file per date subfolder
#     filtered_df.coalesce(1).write \
#         .option("header", "true") \
#         .option("sep", "\t") \
#         .mode("overwrite") \
#         .csv(f"{gcs_path}/date={single_date}")

# print(f"Data successfully written to {gcs_path} with coalesced files in each date subfolder.")

In [ ]:
# Step 1: Add engagement_type to all_engagement_tier_1_df
all_engagement_tier_1_df = all_engagement_tier_1_df.withColumn(
    "engagement_type",
    F.when(col("page_type").isin(["PRODUCT", "UNSUPPORTED", "SEARCH"]), "CLICK")
    .when(col("page_type") == "CART_PRODUCT", "ADD_TO_CART")
    .when(col("page_type") == "CHECKOUT_CONFIRM", "CHECKOUT")
    .otherwise(None)
)

# Step 2: Filter events with non-null search_query_term and page_title for generating negatives
events_with_search = all_engagement_tier_1_df.filter(
    col("search_query_term").isNotNull() &
    col("page_title").isNotNull()
).select("search_query_term").distinct()

# Step 3: Get candidate products (events WITHOUT search query terms)
candidate_products = all_engagement_filtered_ordered_df_with_date.filter(
    col("search_query_term").isNull() &
    col("page_title").isNotNull()
).select("user_id", "page_title").distinct()

In [ ]:
# ============================================
# STEP 1: Multi-GPU Optimized Embedding
# ============================================

unique_search_queries = all_engagement_tier_0_df_cached.filter(col("search_query_term").isNotNull()) \
    .select("search_query_term") \
    .distinct() \
    .repartition(16)  # Increased partitions for better multi-GPU utilization

unique_page_titles = all_engagement_tier_0_df_cached.filter(col("page_title").isNotNull()) \
    .select("page_title") \
    .distinct() \
    .repartition(16)  # Increased partitions for better multi-GPU utilization

embedding_schema = StructType([
    StructField("text", StringType(), False),
    StructField("embedding", ArrayType(FloatType()), False)
])

def compute_embeddings_multi_gpu(iterator):
    """
    Compute embeddings using automatic GPU assignment based on partition ID.
    This function is pickle-safe as it doesn't use threading locks.
    """
    # Get partition ID from task context (unique per partition)
    from pyspark import TaskContext
    context = TaskContext.get()
    partition_id = context.partitionId() if context else 0

    # Total number of GPUs
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1

    # Assign GPU based on partition ID (round-robin)
    gpu_id = partition_id % num_gpus
    device = f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu"

    print(f"Partition {partition_id} using GPU {gpu_id} (device: {device})")

    # Load model on the assigned GPU
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    model = model.to(device)
    model.eval()

    # Process batches
    for batch_df in iterator:
        if batch_df.empty:
            yield batch_df
            continue

        texts = batch_df['text'].tolist()

        # Generate embeddings with optimizations
        with torch.no_grad():
            embeddings = model.encode(
                texts,
                batch_size=128,  # Larger batch size for GPU efficiency
                convert_to_tensor=True,
                device=device,
                show_progress_bar=False,
                normalize_embeddings=True
            )

        # Convert to list of lists for Spark
        embeddings_list = embeddings.cpu().numpy().tolist()

        # Create result DataFrame
        result_df = pd.DataFrame({
            'text': texts,
            'embedding': embeddings_list
        })

        yield result_df

# ============================================
# STEP 2: Compute Embeddings with Multi-GPU
# ============================================

# Search query embeddings with optimal partitioning
search_query_embeddings = unique_search_queries \
    .withColumnRenamed("search_query_term", "text") \
    .repartition(16) \
    .mapInPandas(compute_embeddings_multi_gpu, schema=embedding_schema) \
    .cache()

# Page title embeddings with optimal partitioning
page_title_embeddings = unique_page_titles \
    .withColumnRenamed("page_title", "text") \
    .repartition(16) \
    .mapInPandas(compute_embeddings_multi_gpu, schema=embedding_schema) \
    .cache()

# Trigger computation and print counts
print("=" * 60)
print("Computing embeddings with multi-GPU utilization...")
print("=" * 60)

query_count = search_query_embeddings.count()
page_count = page_title_embeddings.count()

print(f"\n✓ Query embeddings: {query_count:,}")
print(f"✓ Page embeddings: {page_count:,}")
print("=" * 60)

In [ ]:
# ============================================
# 1. Configuration Parameters
# ============================================

# Tier boundaries based on provided distribution statistics
TIER_CONFIG = {
    'very_hard': {
        'min_score': 0.287,  # 95th percentile
        'max_score': 0.388,  # 99th percentile (upper bound for candidate generation)
        'count': 20           # Number of negatives to sample from this tier
    },
    'hard': {
        'min_score': 0.168,  # 75th percentile
        'max_score': 0.287,  # 95th percentile
        'count': 40           # Number of negatives to sample from this tier
    },
    'medium': {
        'min_score': 0.102,  # 50th percentile (median)
        'max_score': 0.168,  # 75th percentile
        'count': 20           # Number of negatives to sample from this tier
    },
    'easy': {
        'min_score': 0.0,    # Minimum
        'max_score': 0.102,  # 50th percentile (median)
        'count': 20           # Number of negatives to sample from this tier
    }
}

# Total negatives per positive = 20 + 40 + 20 + 20 = 100
TOTAL_NEGATIVES_PER_POSITIVE = sum(tier['count'] for tier in TIER_CONFIG.values())

# False negative filtering threshold - only generate candidates below this
FALSE_NEGATIVE_THRESHOLD = 0.388  # 99th percentile

# GPU configuration
NUM_AVAILABLE_GPUS = 2

# Batch processing parameters
QUERY_BATCH_SIZE = 1000
PAGE_CHUNK_SIZE = 10000

MAX_SIMILARITIES_STORED = 20000  # Limit stored pairs per query

# ============================================
# 2. Multi-GPU Similarity Search with Stratified Sampling
# ============================================

def compute_stratified_negatives_multi_gpu(
    query_embeddings: np.ndarray,
    query_ids: List[str],
    page_embeddings_broadcast,
    page_ids: List[str],
    tier_config: Dict,
    false_negative_threshold: float
) -> List[Tuple[str, List[Tuple[str, float, str]]]]:
    """
    Compute stratified hard negatives using multi-GPU setup.

    Args:
        query_embeddings: Query embeddings (n_queries, embedding_dim)
        query_ids: List of query IDs
        page_embeddings_broadcast: Broadcast variable containing page embeddings
        page_ids: List of page IDs
        tier_config: Dictionary defining tier boundaries and sample counts
        false_negative_threshold: Maximum similarity score to consider (99th percentile)

    Returns:
        List of (query_id, [(page_id, similarity_score, tier_name), ...])
    """
    # Get partition ID and determine GPU assignment
    task_context = TaskContext.get()
    partition_id = task_context.partitionId() if task_context else 0
    gpu_id = partition_id % NUM_AVAILABLE_GPUS
    device = torch.device(f'cuda:{gpu_id}')

    # Get page embeddings from broadcast variable
    page_embeddings = page_embeddings_broadcast.value

    # Convert to torch tensors
    page_tensor = torch.tensor(page_embeddings, dtype=torch.float32, device=device)
    page_tensor = Fnn.normalize(page_tensor, p=2, dim=1)

    results = []

    # Process queries in batches
    n_queries = len(query_ids)
    for batch_start in range(0, n_queries, QUERY_BATCH_SIZE):
        batch_end = min(batch_start + QUERY_BATCH_SIZE, n_queries)

        batch_query_embeddings = query_embeddings[batch_start:batch_end]
        batch_query_ids = query_ids[batch_start:batch_end]

        # Convert query batch to tensor
        query_tensor = torch.tensor(batch_query_embeddings, dtype=torch.float32, device=device)
        query_tensor = Fnn.normalize(query_tensor, p=2, dim=1)

        # Process page embeddings in chunks
        n_pages = len(page_ids)

        # Store all similarity pairs for each query in this batch
        batch_similarities = [[] for _ in range(len(batch_query_ids))]

        for chunk_start in range(0, n_pages, PAGE_CHUNK_SIZE):
            chunk_end = min(chunk_start + PAGE_CHUNK_SIZE, n_pages)


            # Get page chunk
            page_chunk = page_tensor[chunk_start:chunk_end]

            # Compute similarities: (batch_size, chunk_size)
            similarities = torch.mm(query_tensor, page_chunk.t())

            # Filter pairs below false negative threshold and store
            for local_query_idx in range(len(batch_query_ids)):

                if len(batch_similarities[local_query_idx]) >= MAX_SIMILARITIES_STORED:
                    continue  # Skip if we already have enough candidates

                query_sims = similarities[local_query_idx].cpu().numpy()

                # Filter by threshold
                valid_mask = query_sims < false_negative_threshold
                valid_indices = np.where(valid_mask)[0]

                for local_page_idx in valid_indices:
                    global_page_idx = chunk_start + local_page_idx
                    sim_score = float(query_sims[local_page_idx])

                    batch_similarities[local_query_idx].append(
                        (page_ids[global_page_idx], sim_score)
                    )

        # Perform stratified sampling for each query
        for local_query_idx, query_id in enumerate(batch_query_ids):
            query_sim_pairs = batch_similarities[local_query_idx]

            if not query_sim_pairs:
                # No valid candidates found
                results.append((query_id, []))
                continue

            # Stratified sampling
            stratified_negatives = []

            for tier_name, tier_params in tier_config.items():
                min_score = tier_params['min_score']
                max_score = tier_params['max_score']
                sample_count = tier_params['count']

                # Filter pairs in this tier
                tier_candidates = [
                    (page_id, score)
                    for page_id, score in query_sim_pairs
                    if min_score <= score < max_score
                ]

                # Sample from tier
                if len(tier_candidates) == 0:
                    # No candidates in this tier - skip or sample from available tiers
                    continue
                elif len(tier_candidates) <= sample_count:
                    # Take all candidates
                    sampled = tier_candidates
                else:
                    # Random sample
                    sampled_indices = np.random.choice(
                        len(tier_candidates),
                        size=sample_count,
                        replace=False
                    )
                    sampled = [tier_candidates[i] for i in sampled_indices]

                # Add tier label
                stratified_negatives.extend([
                    (page_id, score, tier_name)
                    for page_id, score in sampled
                ])

            results.append((query_id, stratified_negatives))

    return results


# ============================================
# 3. PySpark UDF Wrapper
# ============================================

def create_stratified_negative_sampling_udf(page_embeddings_broadcast, page_ids_broadcast):
    """
    Create a UDF for stratified negative sampling.

    Args:
        page_embeddings_broadcast: Broadcast variable for page embeddings
        page_ids_broadcast: Broadcast variable for page IDs

    Returns:
        PySpark UDF
    """

    def stratified_negative_sampling(partition_iter):
        """
        Process a partition of queries and generate stratified negatives.

        Args:
            partition_iter: Iterator of (query_id, embedding_array)

        Yields:
            (query_id, page_id, similarity_score, tier_name)
        """
        # Collect partition data
        partition_data = list(partition_iter)

        if not partition_data:
            return

        query_ids = [row.query_id for row in partition_data]
        query_embeddings = np.array([row.embedding for row in partition_data])

        # Get page data from broadcast
        page_ids = page_ids_broadcast.value

        # Compute stratified negatives
        results = compute_stratified_negatives_multi_gpu(
            query_embeddings=query_embeddings,
            query_ids=query_ids,
            page_embeddings_broadcast=page_embeddings_broadcast,
            page_ids=page_ids,
            tier_config=TIER_CONFIG,
            false_negative_threshold=FALSE_NEGATIVE_THRESHOLD
        )

        # Yield results
        for query_id, negatives in results:
            for page_id, score, tier_name in negatives:
                yield (query_id, page_id, float(score), tier_name)

    return stratified_negative_sampling


# ============================================
# 4. Main Processing Pipeline
# ============================================

def generate_stratified_negatives(
    query_embeddings_df,  # DataFrame with columns: query_id, embedding
    page_embeddings_df,   # DataFrame with columns: page_id, embedding
    num_partitions: int = 200
):
    """
    Generate stratified hard negatives for all queries.

    Args:
        query_embeddings_df: DataFrame with query embeddings
        page_embeddings_df: DataFrame with page embeddings
        num_partitions: Number of partitions for parallel processing

    Returns:
        DataFrame with columns: query_id, page_id, similarity_score, tier_name
    """

    # ============================================
    # Step 1: Broadcast page embeddings
    # ============================================
    print("Broadcasting page embeddings...")

    page_data = page_embeddings_df.select("page_id", "embedding").collect()
    page_ids = [row.page_id for row in page_data]
    page_embeddings = np.array([row.embedding for row in page_data])

    page_embeddings_broadcast = spark.sparkContext.broadcast(page_embeddings)
    page_ids_broadcast = spark.sparkContext.broadcast(page_ids)

    print(f"Broadcast complete. Page embeddings shape: {page_embeddings.shape}")

    # ============================================
    # Step 2: Repartition queries for load balancing
    # ============================================
    query_embeddings_repartitioned = query_embeddings_df.repartition(num_partitions)

    # ============================================
    # Step 3: Apply stratified negative sampling UDF
    # ============================================
    print(f"Starting stratified negative sampling across {num_partitions} partitions...")

    # Define output schema
    output_schema = StructType([
        StructField("query_id", StringType(), False),
        StructField("page_id", StringType(), False),
        StructField("similarity_score", FloatType(), False),
        StructField("tier_name", StringType(), False)
    ])

    # Create UDF
    sampling_udf = create_stratified_negative_sampling_udf(
        page_embeddings_broadcast,
        page_ids_broadcast
    )

    # Apply UDF using mapInPandas or RDD transformation
    negatives_rdd = query_embeddings_repartitioned.rdd.mapPartitions(sampling_udf)

    # Convert back to DataFrame
    negatives_df = spark.createDataFrame(negatives_rdd, schema=output_schema)

    return negatives_df


search_query_embeddings = search_query_embeddings.withColumnRenamed("text", "query_id")
page_title_embeddings = page_title_embeddings.withColumnRenamed("text", "page_id")
# Generate stratified negatives
stratified_negatives_df = generate_stratified_negatives(
    query_embeddings_df=search_query_embeddings,
    page_embeddings_df=page_title_embeddings,
    num_partitions=NUM_AVAILABLE_GPUS * 4  # Scale with available GPUs
)

# ============================================
# 6. Validation Statistics
# ============================================
print("\n" + "=" * 60)
print("VALIDATION STATISTICS")
print("=" * 60)

# Total negatives generated
stratified_negatives_df_cached = stratified_negatives_df.cache()
print(f"Total negatives generated: {stratified_negatives_df_cached.count()}")

# Negatives per query
avg_negatives_per_query = stratified_negatives_df_cached.groupBy("query_id").count() \
    .agg({"count": "avg"}).collect()[0][0]
print(f"Average negatives per query: {avg_negatives_per_query:.2f}")

# Tier distribution
print("\nTier Distribution:")
tier_distribution = stratified_negatives_df_cached.groupBy("tier_name").count().orderBy("tier_name")
tier_distribution.show()

# Similarity score distribution per tier
print("\nSimilarity Score Distribution by Tier:")
stratified_negatives_df_cached.groupBy("tier_name") \
    .agg(
        F.min("similarity_score").alias("min_score"),
        F.max("similarity_score").alias("max_score"),
        F.avg("similarity_score").alias("avg_score")
    ).orderBy("tier_name").show()

# Check for false negatives (should be 0)
false_negatives_count = stratified_negatives_df_cached.filter(
    col("similarity_score") >= FALSE_NEGATIVE_THRESHOLD
).count()
print(f"\nFalse negatives (similarity >= {FALSE_NEGATIVE_THRESHOLD}): {false_negatives_count}")

if false_negatives_count > 0:
    print("WARNING: False negatives detected! Review filtering logic.")

In [ ]:
# stratified_negatives_df_cached = stratified_negatives_df_cached.withColumnRenamed("query_id", "search_query") \
#     .withColumnRenamed("page_id", "page_title").repartition(
#     200,
#     "search_query", "page_title")

In [ ]:
# search_candidate_pairs_truncated = stratified_negatives_df_cached.select('search_query', 'page_title', 'similarity_score').distinct()
# search_candidate_pairs_truncated = search_candidate_pairs_truncated.orderBy(col('search_query').asc(), col('similarity_score').desc())

# # GCS bucket path
# gcs_path = "gs://chanderiyer/projects/honeysearchbenchmarks/datasets/user_engagement_with_negatives/sdata_search_query_page_title_similarity_pairs_dataset2_v1"

# search_candidate_pairs_truncated.write \
#     .option("header", "true") \
#     .option("sep", "\t") \
#     .mode("overwrite") \
#     .csv(f"{gcs_path}")

# print(f"Data successfully written to {gcs_path} with coalesced files in each date subfolder.")

In [ ]:
# GCS bucket path
gcs_path = "gs://chanderiyer/projects/honeysearchbenchmarks/datasets/user_engagement_with_negatives/sdata_search_query_page_title_similarity_pairs_dataset2_v1"
search_candidate_pairs = spark.read.csv(f"{gcs_path}", header=True, sep="\t")
print(f"Data successfully read from {gcs_path}.")

Data successfully read from gs://chanderiyer/projects/honeysearchbenchmarks/datasets/user_engagement_with_negatives/sdata_search_query_page_title_similarity_pairs_dataset2_v1.


In [ ]:
# Continue with joins (fix column names to match your schema)

# Join with explicit condition to avoid duplicate columns
# Create an aliased version of candidate_products first
candidate_products_aliased = candidate_products.select(
    col("page_title").alias("candidate_page_title")
)

# Now join using the aliased DataFrame
search_candidate_pairs_final = search_candidate_pairs.join(
    candidate_products_aliased,
    search_candidate_pairs["page_title"] == candidate_products_aliased["candidate_page_title"],
    how="inner"
).drop("candidate_page_title")  # Drop the aliased column after join


search_candidate_pairs_final = search_candidate_pairs_final.join(
    events_with_search,
    search_candidate_pairs["search_query"] == events_with_search["search_query_term"],  # Fixed reference
    how="inner"
).drop("search_query").select('page_title', 'similarity_score', 'search_query_term').distinct()

In [ ]:
# Popularity score computation
total_users = all_engagement_filtered_ordered_df_with_date.select("user_id").distinct().count()

product_popularity = candidate_products.groupBy("page_title").agg(
    F.countDistinct("user_id").alias("distinct_users")
).withColumn(
    "popularity_score",
    col("distinct_users") / lit(total_users)
).withColumnRenamed("page_title", "pop_page_title")

# Join with explicit condition to avoid duplicate columns
search_candidate_pairs_final = search_candidate_pairs_final.join(
    product_popularity.select(
        col("pop_page_title"),
        col("popularity_score")
    ),
    search_candidate_pairs_final["page_title"] == product_popularity["pop_page_title"],
    how="inner"
).drop("pop_page_title")  # Drop the aliased column after join

search_candidate_pairs_final_combined = search_candidate_pairs_final.withColumn(
    "combined_score",
    col("similarity_score") + col("popularity_score")
).distinct()

In [ ]:
search_candidate_pairs_final_combined_cached = search_candidate_pairs_final_combined.cache()
print(search_candidate_pairs_final_combined_cached.count())

25618181


In [ ]:
search_candidate_pairs_final_combined_cached_aliased = search_candidate_pairs_final_combined_cached.select(
    col("page_title").alias("candidate_page_title"),
    col("search_query_term").alias("candidate_search_query_term"),
    col("combined_score")
)

join_condition = [
    all_engagement_tier_0_df_cached.page_title == search_candidate_pairs_final_combined_cached_aliased.candidate_page_title
]

search_candidate_scored = search_candidate_pairs_final_combined_cached_aliased.join(
    all_engagement_tier_0_df_cached.select(
        col("page_title"),
        col("referrer_url")
    ),
    on=join_condition,
    how='inner'
).select(
    'candidate_search_query_term', 'candidate_page_title', 'referrer_url', 'combined_score'
).distinct()

search_candidate_scored = search_candidate_scored.withColumnRenamed("referrer_url", "candidate_referrer_url")

In [ ]:
join_condition = [
    all_engagement_tier_0_df_cached.search_query_term == search_candidate_scored.candidate_search_query_term
]

search_candidate_scored = all_engagement_tier_0_df_cached.join(
    search_candidate_scored.select(
        col("candidate_page_title"),
        col("candidate_search_query_term"),
        col("candidate_referrer_url"),
        col("combined_score")
    ),
    on=join_condition,
    how='inner'
).select(
    'id', 'ts', 'timestamp', 'storeid', 'storename', 'storecountry',
    'storesessionid', 'screenviewid', 'candidate_page_title', 'page_type', 'user_id',
    'session_id', 'candidate_referrer_url', 'search_query_term', 'combined_score', 'date'
).distinct()

In [ ]:
search_candidate_scored = search_candidate_scored.withColumnRenamed("candidate_page_title", "page_title")
search_candidate_scored = search_candidate_scored.withColumnRenamed("candidate_referrer_url", "referrer_url")
search_candidate_scored = search_candidate_scored.filter(col("search_query_term").isNotNull() & (col("page_title").isNotNull()))
search_candidate_scored = search_candidate_scored.filter(F.length(F.trim(F.col("search_query_term"))) >= 2)

# Step 1: Repartition by key columns to optimize shuffling (adjust partition count based on data size)
# For n1-standard-96: use 192-384 partitions (2-4x the number of cores)
search_candidate_scored = search_candidate_scored.repartition(
    800,
    "search_query_term",
    "page_title"
)

# Step 2: Deduplicate - keep only the latest record for each (search_query_term, page_title)
time_window_spec = Window.partitionBy("search_query_term", "page_title").orderBy(col("ts").desc())

top_negatives = search_candidate_scored \
    .withColumn("rn", row_number().over(time_window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn")

# # Step 4: Get top 10 negatives per (search_query_term, date) by combined_score
# # Fix: Use score_window_spec instead of undefined window_spec
# score_window_spec = Window.partitionBy("search_query_term", "date").orderBy(col("combined_score").desc())

# top_negatives = search_candidate_scored \
#     .withColumn("rank", row_number().over(score_window_spec)) \
#     .filter(col("rank") <= 10) \
#     .drop("rank")  # Drop rank column if not needed downstream

In [ ]:
top_negatives_cached = top_negatives.cache()
print(top_negatives_cached.count())

25618124


In [ ]:
PRODUCT_PAGE_TYPES = ['PRODUCT', 'UNSUPPORTED', 'SEARCH']
ATC_CHECKOUT_CONFIRM_PAGE_TYPES = ['CHECKOUT_CONFIRM', 'CART_PRODUCT']

from pyspark.sql import functions as F

# Filter 1: Product page types with is_search_query_term_derived = 0
filtered_df_1 = all_engagement_tier_1_df.filter(
    (F.col('page_type').isin(PRODUCT_PAGE_TYPES)) &
    (F.col('is_search_query_term_derived') == 0)
)

# Filter 2: ATC/Checkout/Confirm pages with engagement verified OR Amazon storename
filtered_df_2 = all_engagement_tier_1_df.filter(
    (
        (F.col('page_type').isin(ATC_CHECKOUT_CONFIRM_PAGE_TYPES)) &
        (F.col('is_engagement_verified') == 1)
    ) |
    (
        (F.col('page_type').isin(ATC_CHECKOUT_CONFIRM_PAGE_TYPES)) &
        (F.lower(F.col('storename')).contains('amazon'))
    )
)

# Union and drop duplicates
all_engagement_tier_1_df_filtered_df = filtered_df_1.union(filtered_df_2).dropDuplicates()

In [ ]:
# Step 12: Create initial NOCLICK records with required columns
noclick_events_initial = top_negatives.select(
    col("id"),
    col("ts"),
    col("timestamp"),
    col("storeid"),
    col("storename"),
    col("storecountry"),
    col("storesessionid"),
    col("screenviewid"),
    col("page_type"),
    col("user_id"),
    col("session_id"),
    col("referrer_url"),
    col("search_query_term"),
    col("page_title"),
    lit(0).alias("is_engagement_verified"),
    lit(0).alias("is_search_query_term_derived"),
    col("date"),
    lit("NOCLICK").alias("engagement_type")
).dropDuplicates()

# Step 12a: Extract unique (search_query_term, page_title) combinations from engagement events
engaged_combinations = all_engagement_tier_1_df_filtered_df.select(
    "search_query_term",
    "page_title"
).dropDuplicates()

# Step 12b: Filter out NOCLICK events with matching combinations using left anti join
noclick_events = noclick_events_initial.join(
    engaged_combinations,
    on=["search_query_term", "page_title"],
    how="left_anti"
)

# Step 13: Combine with original data
final_user_engagement = all_engagement_tier_1_df_filtered_df.select(
    "id", "ts", "timestamp", "storeid", "storename", "storecountry",
    "storesessionid", "screenviewid", "page_type", "user_id", "session_id",
    "referrer_url", "search_query_term", "page_title",
    "is_engagement_verified", "is_search_query_term_derived", "date",
    "engagement_type"
).unionByName(noclick_events)

# Step 14: Final ordering
final_user_engagement = final_user_engagement.orderBy(
    ["session_id", "user_id", "ts"],
    ascending=False
)

In [ ]:
final_user_engagement_cached = final_user_engagement.cache()
print(final_user_engagement_cached.count())

30026399


In [ ]:
print(final_user_engagement_cached.groupBy("engagement_type").count().show())

+---------------+--------+
|engagement_type|   count|
+---------------+--------+
|        NOCLICK|25617950|
|          CLICK| 3851470|
|    ADD_TO_CART|  310742|
|       CHECKOUT|  246237|
+---------------+--------+

None


In [ ]:
# GCS bucket path
gcs_path = "gs://chanderiyer/projects/honeysearchbenchmarks/datasets/user_engagement_with_negatives/sdata_search_query_derived_tier_1_with_negatives_dataset2_v1"

# Loop through each partition and coalesce into a single file for each date subfolder
for date in final_user_engagement.select("date").distinct().collect():
    single_date = date["date"]
    # Filter by the specific date
    filtered_df = final_user_engagement.filter(F.col("date") == single_date)
    # Coalesce the DataFrame to ensure a single file per date subfolder
    filtered_df.coalesce(1).write \
        .option("header", "true") \
        .option("sep", "\t") \
        .mode("overwrite") \
        .csv(f"{gcs_path}/date={single_date}")

print(f"Data successfully written to {gcs_path} with coalesced files in each date subfolder.")

Data successfully written to gs://chanderiyer/projects/honeysearchbenchmarks/datasets/user_engagement_with_negatives/sdata_search_query_derived_tier_1_with_negatives_dataset2_v1 with coalesced files in each date subfolder.
